# Replication: Cardinality Constrained Multi-Period Mean–Variance Portfolio Optimization

**Paper:** Wang, Jin, Wu, Gao (2026), *Automatica* 183, 112669

**Key methodology:**
1. n=10 Dow Jones assets, T=8 quarterly periods
2. Markov regime-switching model (2 states: bull/bear)
3. CCQO (Cardinality-Constrained Quadratic Optimization) solved via Gurobi MIQP
4. Three strategies compared: CMV-static, CMMV-independent, CMMV
5. Sharpe ratio evaluation with transaction costs

**Note:** The paper's main results (Table 1) use *simulated* data (10⁷ paths) with parameters
from Costa & Araujo (2008). Since we have actual historical CSV data from the group assignment,
we implement the methodology on real data (analogous to the paper's "Empirical" row in Table 2).

## 1. Import Libraries and Set Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
import warnings
warnings.filterwarnings('ignore')

# --- Global Configuration (Paper Section 4) ---
np.random.seed(42)

# Paper parameters
N_ASSETS = 10                    # n = 10 Dow Jones assets
T_QUARTERS = 8                   # T = 8 quarters
Q_DEFAULT = 5                    # Default cardinality constraint
X0 = 1.0                         # Initial wealth
ALPHA_TC = 0.0005                # Transaction cost rate (0.05%, paper Table 1)
ALPHA_MGMT = 0.001               # Management fee rate (0.1%, paper Section 4)
EPSILON = 1e-6                   # Regularization for positive definiteness
BIG_M = 100.0                    # Big-M for MIQP formulation

# Asset names (paper Section 4)
ASSET_NAMES = ['IBM', 'BA', 'MO', 'XOM', 'MMM', 'AIG', 'UTX', 'JNJ', 'PG', 'CAT']
ASSET_LABELS = {
    'IBM': 'IBM (I)', 'BA': 'Boeing (B)', 'MO': 'Altria (A)',
    'XOM': 'Exxon Mobil (E)', 'MMM': '3M (M)', 'AIG': 'AIG (G)',
    'UTX': 'UTC (U)', 'JNJ': 'J&J (J)', 'PG': 'P&G (P)', 'CAT': 'Caterpillar (C)'
}

# Transition probabilities (paper Section 4)
P_TRANS = np.array([
    [0.55, 0.45],  # From state 0 (up): p(0,0)=0.55, p(0,1)=0.45
    [0.55, 0.45]   # From state 1 (down): p(1,0)=0.55, p(1,1)=0.45
])

print("Configuration set successfully.")
print(f"  Assets: {ASSET_NAMES}")
print(f"  Cardinality q={Q_DEFAULT}, Horizon T={T_QUARTERS}")
print(f"  TC rate: {ALPHA_TC}, Management fee rate: {ALPHA_MGMT}")

## 2. Load and Prepare Data

The paper uses parameters calibrated from Costa & Araujo (2008) with data from 2000–2003.
The group assignment uses CSV files with quarterly Dow Jones returns and T-bill rates.
We try to load those CSVs; if unavailable, we download from Yahoo Finance.

In [ ]:
# --- 2. LOAD AND PREPARE DATA ---
# Try loading CSV files from group assignment paths
import os

# Possible CSV paths (adjust as needed for your machine)
csv_paths = [
    # Group member's path
    r"C:\Users\liman\OneDrive - University of Johannesburg\MFE\2026 First Semester\Portfolio Optimisation and Risk Allocation\Group Assignment",
    # Current folder
    os.path.dirname(os.path.abspath("__file__")),
    r"C:\Users\r5582180\OneDrive - FRG\Desktop\Uni\PORAGroupAssignmnet",
]

returns = None
tbill = None

for base_path in csv_paths:
    ret_path = os.path.join(base_path, "dow_returns_2000_2003.csv")
    tb_path = os.path.join(base_path, "us_tbill_3month_2000_2003.csv")
    if os.path.exists(ret_path) and os.path.exists(tb_path):
        returns = pd.read_csv(ret_path, parse_dates=["Date"], index_col="Date")
        tbill = pd.read_csv(tb_path, parse_dates=["Date"], index_col="Date")
        print(f"Loaded data from: {base_path}")
        break

if returns is None:
    print("CSV files not found. Downloading from Yahoo Finance...")
    import yfinance as yf

    # Download quarterly price data for the 10 stocks (2000-2003)
    tickers = ['IBM', 'BA', 'MO', 'XOM', 'MMM', 'AIG', 'UTX', 'JNJ', 'PG', 'CAT']

    # Note: UTX became RTX after merger; use historical data
    prices = yf.download(tickers, start='1999-10-01', end='2003-10-01',
                         interval='3mo', auto_adjust=True)['Close']

    # Compute quarterly total returns (price ratio)
    returns_raw = prices.pct_change().dropna()
    # Keep columns in the right order
    available_cols = [t for t in tickers if t in returns_raw.columns]
    returns = returns_raw[available_cols]

    # Use approximate 3-month T-bill rates for 2000-2003 (quarterly, annualized %)
    # Historical values: ~5.5% in 2000, declining to ~1% by 2003
    tbill_rates = [5.68, 5.76, 6.24, 5.94, 5.13, 3.49, 1.92, 1.72,
                   1.25, 1.11, 1.01, 0.92, 0.93, 0.92, 0.95, 0.94]
    tbill_dates = pd.date_range(start='2000-01-01', periods=len(tbill_rates), freq='QS')
    tbill = pd.DataFrame({'Rate': tbill_rates[:len(returns)]},
                          index=tbill_dates[:len(returns)])
    print(f"Downloaded {len(returns)} quarters of data for {len(returns.columns)} stocks")

# Take first 8 quarters to match paper
returns = returns.iloc[:T_QUARTERS]
tbill = tbill.iloc[:T_QUARTERS]

# Validate dimensions
n = len(returns.columns)
assert n == N_ASSETS or n > 0, f"Expected {N_ASSETS} assets, got {n}"
assert len(returns) == T_QUARTERS, f"Expected {T_QUARTERS} quarters, got {len(returns)}"

print(f"\nData shape: {returns.shape}")
print(f"Columns: {list(returns.columns)}")
print(f"Date range: {returns.index[0]} to {returns.index[-1]}")
print(f"\nFirst few rows of returns:")
print(returns.head())
print(f"\nT-bill rates:")
print(tbill.head())

## 3. Compute Excess Returns and Second Moment Matrix

**Paper equations:**
- Excess return mean: $d = E[P_{t+1}] = \mu - r_{t+1} \cdot e$ (Eq. 8)
- Second moment matrix: $D = \Sigma + d \cdot d^\top$ (Eq. 9)
- Note: D is the *second moment* of excess returns, NOT just the covariance

In [ ]:
# --- 3. COMPUTE EXCESS RETURNS AND SECOND MOMENT MATRIX ---

# Convert T-bill from annual % to quarterly decimal
# Paper: r_t is the risk-free return per period (total return = 1 + r_t)
rf_quarterly = tbill.values.flatten() / 100 / 4  # Annual % -> quarterly decimal

# Compute excess returns: P_{t+1} = R_{t+1} - r_{t+1}
# Paper Eq. 8: d_t(i) = E_t[P_{t+1} | s_t = i] = mu_t(i) - r_{t+1} * e
excess_returns = returns.values - rf_quarterly.reshape(-1, 1)

# Full-sample estimates (for reference)
d_full = excess_returns.mean(axis=0)              # Mean excess return vector
cov_full = np.cov(excess_returns, rowvar=False)    # Covariance matrix of excess returns

# Paper Eq. 9: D = Sigma + d * d^T (second moment of excess returns)
# With regularization for numerical stability
D_full = cov_full + np.outer(d_full, d_full) + EPSILON * np.eye(n)

# Verify positive definiteness
eigenvalues = np.linalg.eigvalsh(D_full)
assert np.all(eigenvalues > 0), f"D is not positive definite! Min eigenvalue: {eigenvalues.min()}"

print(f"d (mean excess return) shape: {d_full.shape}")
print(f"D (second moment matrix) shape: {D_full.shape}")
print(f"D is positive definite: {np.all(eigenvalues > 0)} (min eigenvalue: {eigenvalues.min():.6e})")
print(f"\nMean excess returns (d):")
for name, val in zip(returns.columns, d_full):
    print(f"  {name}: {val:.6f}")
print(f"\nRisk-free rates (quarterly):")
print(f"  {rf_quarterly}")

## 4. Define the CCQO Solver (Gurobi MIQP)

**Paper formulation** (Section 3, Eq. 12 + MIQP in Section 3.1):

$$\min_{k \in \mathbb{R}^n} \quad 2 d^\top k + k^\top D k \quad \text{s.t.} \quad \|k\|_0 \leq q$$

**MIQP equivalent** (big-M formulation):
$$\min_{k, z} \quad 2 d^\top k + k^\top D k$$
$$\text{s.t.} \quad -M z_l \leq k_l \leq M z_l, \quad z_l \in \{0,1\}, \quad \sum z_l \leq q$$

In [ ]:
# --- 4. CCQO SOLVER (Gurobi MIQP) ---
# Paper Section 3.1: P^ccqo_miqp(D_t(i), d_t(i), q)

def solve_ccqo(d_vec, D_mat, q, big_m=None, verbose=False):
    """
    Solve the Cardinality-Constrained Quadratic Optimization (CCQO) problem.

    Paper formulation (Eq. 12):
        min_k  2 * d^T * k + k^T * D * k
        s.t.   ||k||_0 <= q

    MIQP formulation (Section 3.1):
        min_{k,z}  2 * d^T * k + k^T * D * k
        s.t.       -M * z_l <= k_l <= M * z_l,  l = 1,...,n
                   z_l in {0,1},                  l = 1,...,n
                   sum(z_l) <= q

    Parameters
    ----------
    d_vec : array (n,) - mean excess return vector
    D_mat : array (n,n) - second moment matrix (must be PD)
    q : int - maximum number of active assets
    big_m : float - big-M parameter (auto-computed if None)
    verbose : bool - print Gurobi output

    Returns
    -------
    k_opt : array (n,) - optimal allocation vector
    z_opt : array (n,) - binary selection vector
    obj_val : float - optimal objective value
    """
    n_dim = len(d_vec)

    # Auto-compute big-M from unconstrained optimum
    if big_m is None:
        try:
            k_unc = -np.linalg.solve(D_mat, d_vec)
            big_m = max(10.0, 5.0 * np.max(np.abs(k_unc)))
        except np.linalg.LinAlgError:
            big_m = BIG_M

    with gp.Env(empty=True) as env:
        env.setParam('OutputFlag', 1 if verbose else 0)
        env.start()

        with gp.Model("CCQO", env=env) as model:
            # Decision variables
            k = model.addMVar(n_dim, lb=-big_m, ub=big_m, name="k")
            z = model.addMVar(n_dim, vtype=GRB.BINARY, name="z")

            # Objective: min 2*d^T*k + k^T*D*k
            model.setObjective(2 * d_vec @ k + k @ D_mat @ k, GRB.MINIMIZE)

            # Cardinality constraint: sum(z) <= q
            model.addConstr(z.sum() <= q, "cardinality")

            # Big-M linking constraints: -M*z_l <= k_l <= M*z_l
            for l in range(n_dim):
                model.addConstr(k[l] <= big_m * z[l], f"upper_{l}")
                model.addConstr(k[l] >= -big_m * z[l], f"lower_{l}")

            model.optimize()

            if model.status == GRB.OPTIMAL:
                k_opt = k.X
                z_opt = z.X
                return k_opt, z_opt, model.objVal
            else:
                raise RuntimeError(f"Gurobi did not find optimal solution. Status: {model.status}")


def get_portfolio_weights(k_opt, z_opt):
    """
    Convert CCQO solution to portfolio weight proportions.
    The paper's policy is u*_t = r_{t+1} * y_t * k*(i), where k* determines
    the relative allocation. We normalize to get portfolio proportions.
    """
    selected = z_opt > 0.5
    if not np.any(selected):
        return np.zeros(len(k_opt))

    k_selected = k_opt[selected]
    total = np.sum(np.abs(k_selected))

    if total < 1e-12:
        # Equal weight among selected
        weights = np.zeros(len(k_opt))
        weights[selected] = 1.0 / np.sum(selected)
        return weights

    # Normalize by absolute values to get proportions
    weights = np.zeros(len(k_opt))
    weights[selected] = k_selected / total
    return weights


# Quick test
print("Testing CCQO solver...")
k_test, z_test, obj_test = solve_ccqo(d_full, D_full, q=Q_DEFAULT)
w_test = get_portfolio_weights(k_test, z_test)
selected_test = [returns.columns[i] for i in range(n) if z_test[i] > 0.5]
print(f"  Selected {len(selected_test)} assets: {selected_test}")
print(f"  Objective value: {obj_test:.6f}")
print(f"  Weights sum: {w_test.sum():.4f}")
for name, w in zip(returns.columns, w_test):
    if abs(w) > 1e-6:
        print(f"    {name}: {w:.4f}")

## 5. CMV-Static Portfolio (Buy-and-Hold Benchmark)

**Paper definition:** The investor treats all T periods as one period, solves the static CMV
portfolio optimization once using rescaled parameters, then holds until the end.

For our empirical implementation: estimate d and D from the first 2 quarters (training),
solve CCQO once, then hold fixed weights for Q3–Q8 (6 out-of-sample quarters).

In [ ]:
# --- 5. CMV-STATIC PORTFOLIO ---

def run_cmv_static(returns_df, rf_rates, q=Q_DEFAULT, train_end=2):
    """
    CMV-Static benchmark: estimate once from training data, hold for test period.

    Parameters
    ----------
    returns_df : DataFrame (T, n) - quarterly returns
    rf_rates : array (T,) - quarterly risk-free rates
    q : int - cardinality constraint
    train_end : int - number of training quarters

    Returns
    -------
    dict with weights, quarterly returns, selected stocks
    """
    n_assets = returns_df.shape[1]
    T_total = len(returns_df)

    # Training: first train_end quarters
    train_returns = returns_df.iloc[:train_end].values
    train_rf = rf_rates[:train_end]
    train_excess = train_returns - train_rf.reshape(-1, 1)

    # Estimate d and D from training data
    d_train = train_excess.mean(axis=0)
    cov_train = np.cov(train_excess, rowvar=False)
    D_train = cov_train + np.outer(d_train, d_train) + EPSILON * np.eye(n_assets)

    # Solve CCQO once
    k_opt, z_opt, obj = solve_ccqo(d_train, D_train, q=q)
    weights = get_portfolio_weights(k_opt, z_opt)

    selected_idx = np.where(z_opt > 0.5)[0]
    selected_names = [returns_df.columns[i] for i in selected_idx]

    # Compute out-of-sample returns for test period (buy and hold)
    test_returns = []
    for t in range(train_end, T_total):
        ret = np.dot(weights, returns_df.iloc[t].values)
        test_returns.append(ret)

    return {
        'weights': weights,
        'test_returns': np.array(test_returns),
        'selected': selected_names,
        'k_opt': k_opt,
        'z_opt': z_opt,
        'obj': obj,
        'strategy': 'CMV-static'
    }


# Run CMV-static
result_static = run_cmv_static(returns, rf_quarterly, q=Q_DEFAULT, train_end=2)

print("=" * 60)
print("CMV-STATIC (Buy and Hold)")
print("=" * 60)
print(f"Selected assets: {result_static['selected']}")
print(f"Weights:")
for name, w in zip(returns.columns, result_static['weights']):
    if abs(w) > 1e-6:
        print(f"  {name}: {w:.4f}")
print(f"\nOut-of-sample returns (Q3-Q8):")
for t, ret in enumerate(result_static['test_returns']):
    print(f"  Q{t+3}: {ret:.4f} ({ret*100:.2f}%)")
mean_r = result_static['test_returns'].mean()
std_r = result_static['test_returns'].std()
sharpe = mean_r / std_r * 2 if std_r > 0 else 0  # Annualized (quarterly * sqrt(4) = *2)
print(f"\nMean quarterly return: {mean_r:.4f}")
print(f"Std dev quarterly: {std_r:.4f}")
print(f"Sharpe ratio (annualized): {sharpe:.4f}")
print(f"Paper's CMV-static Sharpe (q=5): 0.221")

## 6. CMMV-Independent Portfolio (Expanding Window, No Regime Switching)

**Paper definition (Theorem 11, Assumption 10):** The investor assumes returns are independent
over time. The CCQO is re-solved at each rebalancing point using updated estimates, but there
is no multi-period linkage through regime-switching states.

In [ ]:
# --- 6. CMMV-INDEPENDENT (Expanding Window, Independent Returns) ---

def run_cmmv_independent(returns_df, rf_rates, q=Q_DEFAULT, min_train=2):
    """
    CMMV-Independent: Re-solve CCQO each quarter using expanding window.
    Assumes independent returns (no regime switching, no multi-period linkage).
    This corresponds to Theorem 11 / Assumption 10 in the paper.

    Parameters
    ----------
    returns_df : DataFrame (T, n) - quarterly returns
    rf_rates : array (T,) - quarterly risk-free rates
    q : int - cardinality constraint
    min_train : int - minimum training quarters needed

    Returns
    -------
    dict with per-quarter weights, returns, selections
    """
    n_assets = returns_df.shape[1]
    T_total = len(returns_df)

    all_weights = []
    all_returns = []
    all_selected = []

    for t in range(min_train, T_total):
        # Use all data from Q1 to Q(t-1) for estimation
        train_data = returns_df.iloc[:t].values
        train_rf = rf_rates[:t]
        train_excess = train_data - train_rf.reshape(-1, 1)

        # Estimate d and D
        d_train = train_excess.mean(axis=0)
        cov_train = np.cov(train_excess, rowvar=False)
        D_train = cov_train + np.outer(d_train, d_train) + EPSILON * np.eye(n_assets)

        # Solve CCQO
        k_opt, z_opt, obj = solve_ccqo(d_train, D_train, q=q)
        weights = get_portfolio_weights(k_opt, z_opt)

        selected_idx = np.where(z_opt > 0.5)[0]
        selected_names = [returns_df.columns[i] for i in selected_idx]

        # Out-of-sample return for quarter t
        ret = np.dot(weights, returns_df.iloc[t].values)

        all_weights.append(weights)
        all_returns.append(ret)
        all_selected.append(selected_names)

    return {
        'weights_list': all_weights,
        'test_returns': np.array(all_returns),
        'selected_list': all_selected,
        'strategy': 'CMMV-independent'
    }


# Run CMMV-independent
result_independent = run_cmmv_independent(returns, rf_quarterly, q=Q_DEFAULT)

print("=" * 60)
print("CMMV-INDEPENDENT (Expanding Window, No Regime Switching)")
print("=" * 60)
for t, (ret, sel) in enumerate(zip(result_independent['test_returns'],
                                    result_independent['selected_list'])):
    print(f"  Q{t+3}: Return={ret:.4f} ({ret*100:.2f}%), Selected={sel}")

mean_r = result_independent['test_returns'].mean()
std_r = result_independent['test_returns'].std()
sharpe = mean_r / std_r * 2 if std_r > 0 else 0
print(f"\nMean quarterly return: {mean_r:.4f}")
print(f"Std dev quarterly: {std_r:.4f}")
print(f"Sharpe ratio (annualized): {sharpe:.4f}")
print(f"Paper's CMMV-independent Sharpe (q=5): 0.217")

## 7. CMMV Portfolio (Multi-Period with Regime Switching)

**Paper's CMMV (Theorem 5, Eq. 28):** The dynamic CMMV portfolio leverages the Markov
regime-switching structure. The key variable $\theta_t(s_t)$ is computed recursively backward
(Eq. 10), solving a CCQO sub-problem for each market state at each time.

For the empirical implementation, we classify quarters into bull/bear regimes using a simple
threshold on market returns, estimate regime-conditional parameters, and apply the
multi-period policy.

In [ ]:
# --- 7. CMMV PORTFOLIO (Multi-Period with Regime Switching) ---

def classify_regime(returns_df, threshold=0.0):
    """
    Classify each quarter into bull (0) or bear (1) regime based on
    the average return across all assets.
    """
    avg_returns = returns_df.mean(axis=1)
    states = (avg_returns < threshold).astype(int).values
    return states


def estimate_regime_params(returns_df, rf_rates, states, n_states=2):
    """
    Estimate mean returns and covariance for each regime state.

    Returns
    -------
    d_regime : dict {state: array(n,)} - mean excess returns per regime
    D_regime : dict {state: array(n,n)} - second moment matrix per regime
    """
    n_assets = returns_df.shape[1]
    excess = returns_df.values - rf_rates.reshape(-1, 1)

    d_regime = {}
    D_regime = {}

    for s in range(n_states):
        mask = states == s
        if mask.sum() < 2:
            # Not enough data for this regime; use full-sample estimates
            d_regime[s] = excess.mean(axis=0)
            cov_s = np.cov(excess, rowvar=False)
        else:
            excess_s = excess[mask]
            d_regime[s] = excess_s.mean(axis=0)
            cov_s = np.cov(excess_s, rowvar=False)

        D_regime[s] = cov_s + np.outer(d_regime[s], d_regime[s]) + EPSILON * np.eye(n_assets)

    return d_regime, D_regime


def compute_theta_recursive(d_regime, D_regime, P_trans, q, T_horizon, n_states=2):
    """
    Compute theta recursively backward (Paper Eq. 10).

    theta_T(j) = 1 for all j
    theta_t(i) = (sum_j p(i,j) * theta_{t+1}(j)) * rho(i)

    where rho(i) = 1 - d(i)[z*]^T * D(i)[z*]^{-1} * d(i)[z*]

    Returns
    -------
    theta : array (T_horizon, n_states) - theta values
    k_regime : dict {state: array(n,)} - optimal k for each regime
    z_regime : dict {state: array(n,)} - binary selection for each regime
    """
    n_assets = len(d_regime[0])

    # Solve CCQO for each regime state (time-invariant, Assumption 8)
    k_regime = {}
    z_regime = {}
    rho = np.zeros(n_states)

    for s in range(n_states):
        k_opt, z_opt, obj = solve_ccqo(d_regime[s], D_regime[s], q=q)
        k_regime[s] = k_opt
        z_regime[s] = z_opt

        # Compute rho(i) = 1 - d[z*]^T * D[z*]^{-1} * d[z*]
        selected = z_opt > 0.5
        if np.any(selected):
            d_sel = d_regime[s][selected]
            D_sel = D_regime[s][np.ix_(selected, selected)]
            try:
                rho[s] = 1.0 - d_sel @ np.linalg.solve(D_sel, d_sel)
            except np.linalg.LinAlgError:
                rho[s] = 1.0 - d_sel @ np.linalg.pinv(D_sel) @ d_sel
        else:
            rho[s] = 1.0

    # Backward recursion for theta
    theta = np.ones((T_horizon + 1, n_states))  # theta_T = 1

    for t in range(T_horizon - 1, -1, -1):
        for i in range(n_states):
            # sum_j p(i,j) * theta_{t+1}(j)
            expected_theta = np.sum(P_trans[i, :] * theta[t + 1, :])
            theta[t, i] = expected_theta * rho[i]

    return theta, k_regime, z_regime, rho


def run_cmmv_regime(returns_df, rf_rates, q=Q_DEFAULT, min_train=2, P_trans=P_TRANS):
    """
    CMMV with regime switching: classify regimes, estimate regime-conditional
    parameters, compute theta, and apply the dynamic portfolio policy.
    """
    n_assets = returns_df.shape[1]
    T_total = len(returns_df)

    all_weights = []
    all_returns = []
    all_selected = []
    all_states = []

    for t in range(min_train, T_total):
        # Use expanding window: Q1 to Q(t-1)
        train_data = returns_df.iloc[:t]
        train_rf = rf_rates[:t]

        # Classify regimes in training data
        states = classify_regime(train_data)

        # Estimate regime-conditional parameters
        d_regime, D_regime = estimate_regime_params(train_data, train_rf, states)

        # Compute theta (for remaining horizon)
        remaining_T = T_total - t
        theta, k_regime, z_regime, rho = compute_theta_recursive(
            d_regime, D_regime, P_trans, q, remaining_T
        )

        # Determine current market state for quarter t
        # Use the most recent training quarter's state as proxy
        current_state = states[-1]
        all_states.append(current_state)

        # Get portfolio weights for current regime
        weights = get_portfolio_weights(k_regime[current_state], z_regime[current_state])

        selected_idx = np.where(z_regime[current_state] > 0.5)[0]
        selected_names = [returns_df.columns[i] for i in selected_idx]

        # Out-of-sample return
        ret = np.dot(weights, returns_df.iloc[t].values)

        all_weights.append(weights)
        all_returns.append(ret)
        all_selected.append(selected_names)

    return {
        'weights_list': all_weights,
        'test_returns': np.array(all_returns),
        'selected_list': all_selected,
        'states': all_states,
        'strategy': 'CMMV'
    }


# Run CMMV with regime switching
result_cmmv = run_cmmv_regime(returns, rf_quarterly, q=Q_DEFAULT)

print("=" * 60)
print("CMMV (Multi-Period with Regime Switching)")
print("=" * 60)
for t, (ret, sel, st) in enumerate(zip(result_cmmv['test_returns'],
                                        result_cmmv['selected_list'],
                                        result_cmmv['states'])):
    regime = "Bull" if st == 0 else "Bear"
    print(f"  Q{t+3}: Return={ret:.4f} ({ret*100:.2f}%), Regime={regime}, Selected={sel}")

mean_r = result_cmmv['test_returns'].mean()
std_r = result_cmmv['test_returns'].std()
sharpe = mean_r / std_r * 2 if std_r > 0 else 0
print(f"\nMean quarterly return: {mean_r:.4f}")
print(f"Std dev quarterly: {std_r:.4f}")
print(f"Sharpe ratio (annualized): {sharpe:.4f}")
print(f"Paper's CMMV Sharpe (q=5, simulated): 0.231")
print(f"Paper's CMMV Sharpe (q=5, empirical): 0.211")

## 8. Apply Transaction Costs to All Strategies

**Paper Section 4:**
- Transaction cost: $TC_t = \alpha_1 \sum_i |u_t(i)/P_t(i) - u_{t-1}(i)/P_{t-1}(i)|$, $\alpha_1 = 0.05\%$
- Management fee: $M(x_0, q) = \alpha_0 \cdot q \cdot x_0$, $\alpha_0 = 0.1\%$
- For the weight-based approach: $TC_t \approx \alpha_1 \cdot \sum_i |w_t(i) - w_{t-1}(i)|$ (turnover)

In [ ]:
# --- 8. TRANSACTION COSTS ---

def apply_transaction_costs(result, alpha_tc=ALPHA_TC, alpha_mgmt=ALPHA_MGMT, q=Q_DEFAULT):
    """
    Apply transaction costs and management fees to a strategy's returns.

    Paper Section 4:
    - TC_t = alpha_1 * sum(|w_t - w_{t-1}|)  (turnover-based)
    - Management fee = alpha_0 * q * x_0 (deducted from terminal wealth)

    For CMV-static: TC only at initial purchase (no rebalancing)
    For dynamic strategies: TC at each rebalance
    """
    gross_returns = result['test_returns']
    n_periods = len(gross_returns)

    if result['strategy'] == 'CMV-static':
        # Single set of weights, TC only at purchase
        weights = result['weights']
        prev_w = np.zeros_like(weights)
        turnover = np.sum(np.abs(weights - prev_w))
        tc_initial = alpha_tc * turnover

        net_returns = gross_returns.copy()
        net_returns[0] -= tc_initial
        turnovers = [turnover] + [0.0] * (n_periods - 1)
        tcs = [tc_initial] + [0.0] * (n_periods - 1)
    else:
        # Dynamic strategy: TC at each rebalance
        weights_list = result['weights_list']
        prev_w = np.zeros_like(weights_list[0])
        net_returns = []
        turnovers = []
        tcs = []

        for t in range(n_periods):
            w = weights_list[t]
            turnover = np.sum(np.abs(w - prev_w))
            tc = alpha_tc * turnover
            net_ret = gross_returns[t] - tc
            net_returns.append(net_ret)
            turnovers.append(turnover)
            tcs.append(tc)
            prev_w = w.copy()

        net_returns = np.array(net_returns)

    # Management fee (deducted proportionally)
    mgmt_fee = alpha_mgmt * q  # Total management fee
    mgmt_per_period = mgmt_fee / n_periods  # Spread across periods
    net_returns_with_mgmt = net_returns - mgmt_per_period

    # Compute wealth paths
    wealth_gross = np.cumprod(1 + gross_returns)
    wealth_net = np.cumprod(1 + net_returns)
    wealth_net_mgmt = np.cumprod(1 + net_returns_with_mgmt)

    return {
        'gross_returns': gross_returns,
        'net_returns': net_returns,
        'net_returns_with_mgmt': net_returns_with_mgmt,
        'turnovers': turnovers,
        'tcs': tcs,
        'total_tc': sum(tcs),
        'mgmt_fee': mgmt_fee,
        'wealth_gross': wealth_gross,
        'wealth_net': wealth_net,
        'wealth_net_mgmt': wealth_net_mgmt,
    }


# Apply TC to all strategies
tc_static = apply_transaction_costs(result_static)
tc_independent = apply_transaction_costs(result_independent)
tc_cmmv = apply_transaction_costs(result_cmmv)

# Print TC summary
print("=" * 60)
print("TRANSACTION COST SUMMARY")
print("=" * 60)
for name, tc_result in [("CMV-static", tc_static),
                          ("CMMV-independent", tc_independent),
                          ("CMMV", tc_cmmv)]:
    print(f"\n{name}:")
    print(f"  Total TC: {tc_result['total_tc']:.6f}")
    print(f"  Management fee: {tc_result['mgmt_fee']:.4f}")
    for t in range(len(tc_result['turnovers'])):
        print(f"  Q{t+3}: Turnover={tc_result['turnovers'][t]:.4f}, TC={tc_result['tcs'][t]:.6f}")

## 9. Performance Metrics Summary

In [ ]:
# --- 9. PERFORMANCE METRICS ---

def compute_metrics(returns_arr, label=""):
    """Compute standard portfolio performance metrics."""
    mean_r = returns_arr.mean()
    std_r = returns_arr.std()
    sharpe_q = mean_r / std_r if std_r > 0 else 0
    sharpe_ann = sharpe_q * 2  # Annualized for quarterly data (sqrt(4)=2)
    wealth = np.prod(1 + returns_arr)
    return {
        'label': label,
        'mean_return': mean_r,
        'std_return': std_r,
        'sharpe_quarterly': sharpe_q,
        'sharpe_annualized': sharpe_ann,
        'terminal_wealth': wealth,
    }

# Compute metrics for all strategies
metrics_rows = []
for name, tc_res in [("CMV-static", tc_static),
                       ("CMMV-independent", tc_independent),
                       ("CMMV", tc_cmmv)]:
    m_gross = compute_metrics(tc_res['gross_returns'], f"{name} (No TC)")
    m_net = compute_metrics(tc_res['net_returns'], f"{name} (With TC)")
    m_net_mgmt = compute_metrics(tc_res['net_returns_with_mgmt'], f"{name} (TC+Mgmt)")
    m_gross['total_tc'] = tc_res['total_tc']
    m_net['total_tc'] = tc_res['total_tc']
    m_net_mgmt['total_tc'] = tc_res['total_tc']
    metrics_rows.extend([m_gross, m_net, m_net_mgmt])

metrics_df = pd.DataFrame(metrics_rows)
metrics_df = metrics_df.set_index('label')

print("=" * 80)
print("PERFORMANCE METRICS SUMMARY")
print("=" * 80)
print(metrics_df[['mean_return', 'std_return', 'sharpe_annualized',
                   'terminal_wealth', 'total_tc']].to_string(float_format='{:.4f}'.format))

# Wealth path plot
fig, ax = plt.subplots(figsize=(10, 6))
quarters = [f"Q{i}" for i in range(3, 3 + len(tc_static['wealth_gross']))]
ax.plot(quarters, tc_static['wealth_gross'], 'r-s', label='CMV-static', linewidth=2)
ax.plot(quarters, tc_independent['wealth_gross'], 'g-^', label='CMMV-independent', linewidth=2)
ax.plot(quarters, tc_cmmv['wealth_gross'], 'b-o', label='CMMV', linewidth=2)
ax.set_xlabel('Quarter')
ax.set_ylabel('Cumulative Wealth')
ax.set_title('Wealth Trajectories (Gross, No TC)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Reproduce Paper Table 1: Sharpe Ratio Comparison (q=5)

In [ ]:
# --- 10. REPRODUCE TABLE 1 ---

# Paper Table 1 values (q=5, simulated data)
paper_table1 = {
    'CMV-static':       {'TC': 0,       'SR': 0.221},
    'CMMV-independent': {'TC': 5.005e-4, 'SR': 0.217},
    'CMMV':             {'TC': 9.050e-4, 'SR': 0.231},
}

# Our results
our_results = {}
for name, tc_res in [("CMV-static", tc_static),
                       ("CMMV-independent", tc_independent),
                       ("CMMV", tc_cmmv)]:
    gross = tc_res['gross_returns']
    net = tc_res['net_returns']
    sr_gross = gross.mean() / gross.std() * 2 if gross.std() > 0 else 0
    sr_net = net.mean() / net.std() * 2 if net.std() > 0 else 0
    our_results[name] = {
        'TC': tc_res['total_tc'],
        'SR_gross': sr_gross,
        'SR_net': sr_net
    }

# Print comparison table
print("=" * 80)
print("TABLE 1 COMPARISON: Transaction Cost and Sharpe Ratio (q=5)")
print("=" * 80)
print(f"{'Model':<25} {'Our TC':<12} {'Paper TC':<12} {'Our SR(no TC)':<15} {'Our SR(TC)':<15} {'Paper SR':<12}")
print("-" * 91)
for name in ['CMV-static', 'CMMV-independent', 'CMMV']:
    our = our_results[name]
    paper = paper_table1[name]
    print(f"{name:<25} {our['TC']:<12.6f} {paper['TC']:<12.6f} {our['SR_gross']:<15.4f} {our['SR_net']:<15.4f} {paper['SR']:<12.3f}")

print("\n" + "=" * 80)
print("NOTES:")
print("- Paper uses 10^7 SIMULATED sample paths from Markov model (Costa & Araujo 2008)")
print("- Our results use ACTUAL historical data (empirical approach)")
print("- Deviations are expected; paper's Table 2 shows empirical SR can differ significantly")
print(f"  Paper Table 2 empirical: CMV-static=0.151, CMMV-indep=0.133, CMMV=0.211")
print("=" * 80)

## 11. Reproduce Paper Figure 3: Sharpe Ratio vs Time Horizon

In [ ]:
# --- 11. REPRODUCE FIGURE 3: Sharpe vs Horizon ---

horizons = list(range(2, min(8, T_QUARTERS)))  # T=2 to 7
sharpe_by_horizon = {'CMV-static': [], 'CMMV-independent': [], 'CMMV': []}

for T in horizons:
    if T + 2 > T_QUARTERS:
        for key in sharpe_by_horizon:
            sharpe_by_horizon[key].append(np.nan)
        continue

    # --- CMV-static ---
    res_s = run_cmv_static(returns, rf_quarterly, q=Q_DEFAULT, train_end=T)
    # Only use next 2 quarters for testing
    test_rets = res_s['test_returns'][:2]
    if len(test_rets) >= 2 and np.std(test_rets) > 0:
        sharpe_by_horizon['CMV-static'].append(test_rets.mean() / test_rets.std() * 2)
    else:
        sharpe_by_horizon['CMV-static'].append(np.nan)

    # --- CMMV-independent ---
    # Re-estimate at each of the 2 test quarters using expanding window
    ind_rets = []
    for t_test in range(T, min(T + 2, T_QUARTERS)):
        train_d = returns.iloc[:t_test].values
        train_r = rf_quarterly[:t_test]
        excess_t = train_d - train_r.reshape(-1, 1)
        d_t = excess_t.mean(axis=0)
        cov_t = np.cov(excess_t, rowvar=False)
        D_t = cov_t + np.outer(d_t, d_t) + EPSILON * np.eye(n)
        k_t, z_t, _ = solve_ccqo(d_t, D_t, q=Q_DEFAULT)
        w_t = get_portfolio_weights(k_t, z_t)
        ind_rets.append(np.dot(w_t, returns.iloc[t_test].values))
    ind_rets = np.array(ind_rets)
    if len(ind_rets) >= 2 and np.std(ind_rets) > 0:
        sharpe_by_horizon['CMMV-independent'].append(ind_rets.mean() / ind_rets.std() * 2)
    else:
        sharpe_by_horizon['CMMV-independent'].append(np.nan)

    # --- CMMV (with regime switching) ---
    cmmv_rets = []
    for t_test in range(T, min(T + 2, T_QUARTERS)):
        train_df = returns.iloc[:t_test]
        train_r = rf_quarterly[:t_test]
        states_t = classify_regime(train_df)
        d_reg, D_reg = estimate_regime_params(train_df, train_r, states_t)
        remaining = T_QUARTERS - t_test
        _, k_reg, z_reg, _ = compute_theta_recursive(d_reg, D_reg, P_TRANS, Q_DEFAULT, remaining)
        curr_state = states_t[-1]
        w_c = get_portfolio_weights(k_reg[curr_state], z_reg[curr_state])
        cmmv_rets.append(np.dot(w_c, returns.iloc[t_test].values))
    cmmv_rets = np.array(cmmv_rets)
    if len(cmmv_rets) >= 2 and np.std(cmmv_rets) > 0:
        sharpe_by_horizon['CMMV'].append(cmmv_rets.mean() / cmmv_rets.std() * 2)
    else:
        sharpe_by_horizon['CMMV'].append(np.nan)

# Plot Figure 3
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(horizons, sharpe_by_horizon['CMV-static'], 'r-s', linewidth=2, markersize=8,
        label='CMV-static', markerfacecolor='red')
ax.plot(horizons, sharpe_by_horizon['CMMV-independent'], 'g-^', linewidth=2, markersize=8,
        label='CMMV-independent', markerfacecolor='green')
ax.plot(horizons, sharpe_by_horizon['CMMV'], 'b-o', linewidth=2, markersize=8,
        label='CMMV', markerfacecolor='blue')
ax.set_xlabel('Training Horizon T (quarters)', fontsize=14)
ax.set_ylabel('Sharpe Ratio (Annualized)', fontsize=14)
ax.set_title('Figure 3: Sharpe Ratio vs Time Horizon (q=5)', fontsize=16)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Data table
print("\nFigure 3 Data:")
print(f"{'T':<6} {'CMV-static':<15} {'CMMV-indep':<15} {'CMMV':<15}")
print("-" * 51)
for i, T in enumerate(horizons):
    vals = [sharpe_by_horizon[k][i] for k in ['CMV-static', 'CMMV-independent', 'CMMV']]
    print(f"{T:<6} {vals[0]:<15.4f} {vals[1]:<15.4f} {vals[2]:<15.4f}")
print("\nPaper's key finding: CMMV Sharpe increases faster with T than benchmarks.")

## 12. Sensitivity Analysis: Varying Cardinality q

In [ ]:
# --- 12. SENSITIVITY: Varying q ---

q_values = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
sharpe_by_q = {'CMV-static': [], 'CMMV-independent': [], 'CMMV': []}
tc_by_q = {'CMV-static': [], 'CMMV-independent': [], 'CMMV': []}

for q in q_values:
    # CMV-static
    try:
        res_s = run_cmv_static(returns, rf_quarterly, q=q, train_end=2)
        tc_s = apply_transaction_costs(res_s, q=q)
        sr_s = tc_s['net_returns_with_mgmt'].mean() / tc_s['net_returns_with_mgmt'].std() * 2 \
               if tc_s['net_returns_with_mgmt'].std() > 0 else 0
        sharpe_by_q['CMV-static'].append(sr_s)
        tc_by_q['CMV-static'].append(tc_s['total_tc'])
    except Exception:
        sharpe_by_q['CMV-static'].append(np.nan)
        tc_by_q['CMV-static'].append(np.nan)

    # CMMV-independent
    try:
        res_i = run_cmmv_independent(returns, rf_quarterly, q=q)
        tc_i = apply_transaction_costs(res_i, q=q)
        sr_i = tc_i['net_returns_with_mgmt'].mean() / tc_i['net_returns_with_mgmt'].std() * 2 \
               if tc_i['net_returns_with_mgmt'].std() > 0 else 0
        sharpe_by_q['CMMV-independent'].append(sr_i)
        tc_by_q['CMMV-independent'].append(tc_i['total_tc'])
    except Exception:
        sharpe_by_q['CMMV-independent'].append(np.nan)
        tc_by_q['CMMV-independent'].append(np.nan)

    # CMMV
    try:
        res_c = run_cmmv_regime(returns, rf_quarterly, q=q)
        tc_c = apply_transaction_costs(res_c, q=q)
        sr_c = tc_c['net_returns_with_mgmt'].mean() / tc_c['net_returns_with_mgmt'].std() * 2 \
               if tc_c['net_returns_with_mgmt'].std() > 0 else 0
        sharpe_by_q['CMMV'].append(sr_c)
        tc_by_q['CMMV'].append(tc_c['total_tc'])
    except Exception:
        sharpe_by_q['CMMV'].append(np.nan)
        tc_by_q['CMMV'].append(np.nan)

# Plot Figure 4: SR vs q with costs
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(q_values, sharpe_by_q['CMV-static'], 'r-s', linewidth=2, markersize=8,
        label='CMV-static', markerfacecolor='red')
ax.plot(q_values, sharpe_by_q['CMMV-independent'], 'g-^', linewidth=2, markersize=8,
        label='CMMV-independent', markerfacecolor='green')
ax.plot(q_values, sharpe_by_q['CMMV'], 'b-o', linewidth=2, markersize=8,
        label='CMMV', markerfacecolor='blue')
ax.set_xlabel('Cardinality Constraint q', fontsize=14)
ax.set_ylabel('Sharpe Ratio (Annualized, with TC+Mgmt)', fontsize=14)
ax.set_title('Figure 4: Sharpe Ratio vs Cardinality q (with costs)', fontsize=16)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xticks(q_values)
plt.tight_layout()
plt.show()

# Print table (matches paper Table 1 format)
print("\nTable 1 Replication: TC and SR for varying q")
print(f"{'q':<4} {'CMV-static':<20} {'CMMV-independent':<25} {'CMMV':<20}")
print(f"{'':4} {'TC':>8} {'SR':>8}   {'TC':>10} {'SR':>10}   {'TC':>8} {'SR':>8}")
print("-" * 70)
for i, q in enumerate(q_values):
    print(f"{q:<4} {tc_by_q['CMV-static'][i]:>8.6f} {sharpe_by_q['CMV-static'][i]:>8.3f}   "
          f"{tc_by_q['CMMV-independent'][i]:>10.6f} {sharpe_by_q['CMMV-independent'][i]:>10.3f}   "
          f"{tc_by_q['CMMV'][i]:>8.6f} {sharpe_by_q['CMMV'][i]:>8.3f}")

## 13. Audit of Existing Notebook (GROUP PORA ASS2.ipynb)

Systematic comparison of the group assignment notebook against the paper's methodology.

In [ ]:
# --- 13. AUDIT OF EXISTING NOTEBOOK ---

audit_findings = """
╔══════════════════════════════════════════════════════════════════════════╗
║                    AUDIT: GROUP PORA ASS2.ipynb                        ║
╚══════════════════════════════════════════════════════════════════════════╝

FINDING 1: MISSING solve_ccqo_exact FUNCTION DEFINITION
  Severity: CRITICAL (code cannot run)
  Location: Cell 6 calls solve_ccqo_exact() but it is never defined
  Impact:   Entire notebook fails at Cell 6
  Fix:      Add the CCQO solver cell (our Section 4) before Cell 6

FINDING 2: NO REGIME-SWITCHING MODEL
  Severity: MAJOR (fundamental methodology gap)
  Location: All cells (Cells 6-11)
  Paper:    CMMV uses Markov regime-switching with 2 states (Eq. 10, 28)
  Notebook: Treats all data as single regime, no state classification
  Impact:   "CMMV" in notebook is actually just CMMV-independent
  Fix:      Implement regime classification + regime-conditional parameters

FINDING 3: CMMV = CMMV-INDEPENDENT (identical implementation)
  Severity: MAJOR
  Location: Cell 6 ("CMMV rolling window") vs Cell 9 ("CMMV-independent")
  Paper:    CMMV (Theorem 5) differs from CMMV-independent (Theorem 11)
            by using regime-dependent k*(i) vectors
  Notebook: Both use same logic (expanding window + single CCQO solve)
  Impact:   Cannot show CMMV superiority over CMMV-independent
  Fix:      Cell 6 should use regime-conditional d(i), D(i)

FINDING 4: SIMULATION vs EMPIRICAL CONFUSION
  Severity: MAJOR
  Location: Cell 3 (data loading), Cell 9 (comparison table)
  Paper:    Main results (Table 1) use 10^7 SIMULATED sample paths
            Empirical backtesting (Table 2) uses rolling window on real data
  Notebook: Uses 8 quarters of actual data for everything
  Impact:   Results are empirical but compared to paper's simulation results
  Fix:      Either implement simulation OR compare to paper's Table 2 empirical row

FINDING 5: D MATRIX CONSTRUCTION IS CORRECT
  Severity: OK
  Location: Cell 4
  Paper:    D = Σ + d·d^T (Eq. 9)
  Notebook: D = cov_matrix + np.outer(d, d) + epsilon*I ← correct
  Note:     Regularization (epsilon*I) is a reasonable addition

FINDING 6: SHARPE RATIO ANNUALIZATION
  Severity: MINOR
  Location: Cells 6, 9-11
  Paper:    SR = sqrt(1/θ - 1) analytically, or empirical mean/std
  Notebook: Uses mean/std * 2 (annualized) ← correct for quarterly data
  Cell 10:  Uses np.sqrt(4) in one place and *2 elsewhere ← inconsistent
  Fix:      Standardize to *2 (= sqrt(4)) everywhere

FINDING 7: HARDCODED SHARPE VALUES IN COMPARISON TABLE
  Severity: MODERATE
  Location: Cell 9 (final comparison table)
  Code:     print(f"CMV-static (No TC)  {-0.1623:<15.4f}  {0.221:<15}")
            print(f"CMMV (No TC)        {0.5843:<15.4f}  {0.231:<15}")
  Problem:  Sharpe values are hardcoded, not computed dynamically
  Impact:   Table doesn't update when data or parameters change
  Fix:      Use computed variables instead of literals

FINDING 8: TRANSACTION COST MODEL IS SIMPLIFIED
  Severity: MINOR
  Location: Cell 7
  Paper:    TC = α₁ · Σ|u_t(i)/P_t(i) - u_{t-1}(i)/P_{t-1}(i)| (shares-based)
            + Management fee: M = α₀ · q · x₀
  Notebook: TC = α · Σ|w_t(i) - w_{t-1}(i)| (weight-based turnover)
            No management fee
  Impact:   TC is approximated; management fee missing
  Fix:      Add management fee deduction; consider shares-based TC

FINDING 9: HARDCODED DIMENSION (n=10)
  Severity: MINOR
  Location: Cell 7: prev_weights = np.zeros(10)
  Problem:  Should use np.zeros(n) for generality
  Fix:      Replace 10 with n

FINDING 10: DATA PATH PORTABILITY
  Severity: MODERATE
  Location: Cell 3
  Problem:  Hardcoded path to another user's OneDrive folder
  Fix:      Use relative paths or environment variable

═══════════════════════════════════════════════════════════════════════════
SUMMARY: 2 CRITICAL, 3 MAJOR, 2 MODERATE, 3 MINOR findings
═══════════════════════════════════════════════════════════════════════════
"""

print(audit_findings)